# Linear Regression from Scratch 🚀
### Using Gradient Descent on the Automobile Dataset

In this notebook, we implement **Linear Regression** from scratch — no `sklearn`, just pure math and NumPy.

We cover:
1. **Single Variable Linear Regression** (Horsepower → MPG)
2. **Multiple Variable Linear Regression** (Horsepower + Weight + Displacement → MPG)
3. **Vectorized implementation** (cleaner, faster version using NumPy)


## 📦 Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 📂 Load the Dataset

In [ ]:
df = pd.read_csv('Automobile.csv')
df.head()

---
## Part 1 — Single Variable Linear Regression

We want to predict **MPG** (miles per gallon) from **Horsepower**.

The model equation is:

$$\hat{y} = \theta_0 + \theta_1 \cdot x$$

We learn $\theta_0$ and $\theta_1$ using **Gradient Descent**, which minimizes the **Mean Squared Error (MSE)**:

$$MSE = \frac{1}{2n} \sum_{i=1}^{n} (\hat{y}_i - y_i)^2$$

### 🔧 Data Cleaning & Feature Scaling

**Feature Scaling** (Min-Max Normalization) maps values to the range [0, 1].
This helps gradient descent converge faster and more stably.

$$x_{scaled} = \frac{x - x_{min}}{x_{max} - x_{min}}$$

In [ ]:
# Convert horsepower to numeric and drop rows with missing values
df['horsepower'] = pd.to_numeric(df['horsepower'], errors='coerce')
df = df.dropna(subset=['horsepower'])

# Min-Max feature scaling for horsepower
df['horsepower_FS'] = (df['horsepower'] - df['horsepower'].min()) / \
                      (df['horsepower'].max() - df['horsepower'].min())

print(f"Horsepower range: {df['horsepower'].min()} – {df['horsepower'].max()}")
print(f"Scaled range:     {df['horsepower_FS'].min():.2f} – {df['horsepower_FS'].max():.2f}")

### ⚙️ Gradient Descent — Single Variable

At each iteration we compute the gradients and update the parameters:

$$\frac{\partial}{\partial \theta_1} = \frac{1}{n} \sum (\hat{y}_i - y_i) \cdot x_i$$

$$\frac{\partial}{\partial \theta_0} = \frac{1}{n} \sum (\hat{y}_i - y_i)$$

$$\theta_j := \theta_j - \alpha \cdot \frac{\partial}{\partial \theta_j}$$

Where $\alpha$ is the **learning rate**.

In [ ]:
# Initialize parameters
theta0 = 0
theta1 = 0
alpha = 0.9   # learning rate

# Gradient Descent loop
for i in range(1000):
    df['y_'] = (theta1 * df['horsepower_FS']) + theta0
    error = df['y_'] - df['mpg']
    mse = (error**2).sum() / (2 * len(df))

    if i % 100 == 0:
        print(f"Iteration {i:>4}: MSE = {mse:.4f}")

    gradient_theta1 = (error * df['horsepower_FS']).sum() / len(df)
    theta1 = theta1 - (alpha * gradient_theta1)

    gradient_theta0 = error.sum() / len(df)
    theta0 = theta0 - (alpha * gradient_theta0)

print(f"\nFinal parameters:  theta0 = {theta0:.4f},  theta1 = {theta1:.4f}")

### 📊 Plot the Results

In [ ]:
plt.figure(figsize=(9, 5))
plt.scatter(df['horsepower_FS'], df['mpg'], color='steelblue', alpha=0.5, label='Actual Data')
plt.plot(df['horsepower_FS'], df['y_'], color='red', linewidth=2.5, label='Gradient Descent Line')
plt.xlabel('Horsepower (Scaled)')
plt.ylabel('MPG')
plt.title('Single Variable Linear Regression — Gradient Descent 🚀')
plt.legend()
plt.tight_layout()
plt.show()

### 🔮 Predictions on New Cars

In [ ]:
def predict_mpg_single(horsepower):
    """Predict MPG for a car given its horsepower."""
    hp_scaled = (horsepower - df['horsepower'].min()) / \
                (df['horsepower'].max() - df['horsepower'].min())
    return (theta1 * hp_scaled) + theta0

for hp in [200, 300]:
    mpg = predict_mpg_single(hp)
    print(f"Predicted MPG for a car with {hp} HP: {mpg:.2f}")

---
## Part 2 — Multiple Variable Linear Regression

Now we use **3 features** to predict MPG:
- Horsepower
- Weight
- Displacement

The model becomes:

$$\hat{y} = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \theta_3 x_3$$

More features generally means better predictions!

### 🔧 Feature Scaling for All Features

In [ ]:
# Scale weight and displacement the same way
df['weight_FS'] = (df['weight'] - df['weight'].min()) / \
                  (df['weight'].max() - df['weight'].min())

df['displacement_FS'] = (df['displacement'] - df['displacement'].min()) / \
                        (df['displacement'].max() - df['displacement'].min())

print("Features scaled successfully.")
df[['horsepower_FS', 'weight_FS', 'displacement_FS']].describe().round(3)

### ⚙️ Gradient Descent — Multiple Variables (Loop Version)

We now have 4 parameters to update at each step: $\theta_0, \theta_1, \theta_2, \theta_3$.

In [ ]:
# Build feature DataFrame with a bias column
features = df[['horsepower_FS', 'weight_FS', 'displacement_FS']].copy()
features['bias'] = 1

# Initialize parameters as a list
thetaV = [0, 0, 0, 0]   # [theta0, theta1, theta2, theta3]
alpha = 0.5

for i in range(1000):
    features['y_'] = (thetaV[0] * features['bias']) + \
                     (thetaV[1] * features['horsepower_FS']) + \
                     (thetaV[2] * features['weight_FS']) + \
                     (thetaV[3] * features['displacement_FS'])

    error = features['y_'] - df['mpg']
    mse = (error**2).sum() / (2 * len(df))

    if i % 100 == 0:
        print(f"Iteration {i:>4}: MSE = {mse:.4f}")

    # Compute gradients
    gradient_theta0 = error.sum() / len(df)
    gradient_theta1 = (error * features['horsepower_FS']).sum() / len(features)
    gradient_theta2 = (error * features['weight_FS']).sum() / len(features)
    gradient_theta3 = (error * features['displacement_FS']).sum() / len(features)

    # Simultaneous update
    thetaV[0] -= alpha * gradient_theta0
    thetaV[1] -= alpha * gradient_theta1
    thetaV[2] -= alpha * gradient_theta2
    thetaV[3] -= alpha * gradient_theta3

print(f"\nTheta0 (Bias)        = {thetaV[0]:.4f}")
print(f"Theta1 (Horsepower)  = {thetaV[1]:.4f}")
print(f"Theta2 (Weight)      = {thetaV[2]:.4f}")
print(f"Theta3 (Displacement)= {thetaV[3]:.4f}")

---
## Part 3 — Vectorized Implementation ⚡ (Cleaner & Faster)

Instead of computing each gradient separately, we use **matrix operations**.

In matrix form:

$$\hat{y} = X \cdot \Theta$$

$$\text{Gradient} = \frac{1}{n} X^T \cdot (\hat{y} - y)$$

$$\Theta := \Theta - \alpha \cdot \text{Gradient}$$

This computes ALL gradients in a **single line** — much cleaner and efficient!

In [ ]:
n = len(df)

# Build the design matrix X: shape (n, 4) — bias + 3 features
X = np.hstack([
    np.ones((n, 1)),                              # bias column
    df['horsepower_FS'].values.reshape(-1, 1),
    df['weight_FS'].values.reshape(-1, 1),
    df['displacement_FS'].values.reshape(-1, 1)
])

# Target vector
y = df['mpg'].values.reshape(-1, 1)   # shape (n, 1)

# Initialize parameter vector
Theta = np.zeros((4, 1))              # shape (4, 1)

alpha = 0.5

for i in range(1000):
    y_pred = X.dot(Theta)             # (n, 1)
    error  = y_pred - y               # (n, 1)
    mse    = (error**2).sum() / (2 * n)

    if i % 100 == 0:
        print(f"Iteration {i:>4}: MSE = {mse:.4f}")

    # All 4 gradients computed in ONE line!
    gradient = np.dot(X.T, error) / n    # (4, 1)
    Theta    = Theta - (alpha * gradient)

print(f"\nTheta0 (Bias)        = {Theta[0][0]:.4f}")
print(f"Theta1 (Horsepower)  = {Theta[1][0]:.4f}")
print(f"Theta2 (Weight)      = {Theta[2][0]:.4f}")
print(f"Theta3 (Displacement)= {Theta[3][0]:.4f}")

### 🔮 Predictions on New Cars — Multiple Features

In [ ]:
# New car data: [horsepower, weight, displacement]
new_cars_data = np.array([
    [ 65.0,  1800.0,  90.0],   # small efficient car
    [130.0,  3100.0, 180.0],   # mid-size sedan
    [300.0,  4800.0, 400.0]    # large powerful car
])

# Scale the new data using the same min/max from training
hp_scaled   = (new_cars_data[:, 0] - df['horsepower'].min()) / (df['horsepower'].max() - df['horsepower'].min())
w_scaled    = (new_cars_data[:, 1] - df['weight'].min())     / (df['weight'].max()     - df['weight'].min())
disp_scaled = (new_cars_data[:, 2] - df['displacement'].min()) / (df['displacement'].max() - df['displacement'].min())

# Build the design matrix for new cars
X_new = np.hstack([
    np.ones((3, 1)),
    hp_scaled.reshape(-1, 1),
    w_scaled.reshape(-1, 1),
    disp_scaled.reshape(-1, 1)
])

predictions = X_new.dot(Theta)

labels = ['Small car (65 HP, 1800 lbs)', 'Mid-size (130 HP, 3100 lbs)', 'Large car (300 HP, 4800 lbs)']
for label, pred in zip(labels, predictions.flatten()):
    print(f"{label:40s} → Predicted MPG: {pred:.2f}")

---
## 📝 Summary

| Part | Features Used | Final MSE |
|------|:-------------|----------:|
| Single variable | Horsepower only | higher |
| Multiple variable (loop) | HP + Weight + Displacement | lower |
| Multiple variable (vectorized) | HP + Weight + Displacement | same as above ✅ |

**Key takeaways:**
- Feature scaling is essential for gradient descent to work properly.
- More relevant features → lower MSE → better predictions.
- The vectorized (NumPy matrix) approach is equivalent to the loop version but far more concise and efficient.
- Always scale new data using the **same min/max values** from the training set.